<a href="https://colab.research.google.com/github/Trixxzer/PrajwolShrestha_2418111_AIML_Worksheets/blob/main/Worksheet6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning

In [8]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras import Sequential

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train = "/content/drive/MyDrive/AI-ML/FruitinAmazon/train"
test = "/content/drive/MyDrive/AI-ML/FruitinAmazon/test"

In [5]:
os.listdir("/content/drive/MyDrive/AI-ML/FruitinAmazon/test")

['cupuacu', 'acai', 'pupunha', 'graviola', 'tucuma', 'guarana']

In [12]:
import os
from PIL import Image

dataset_path = "/content/drive/MyDrive/AI-ML/FruitinAmazon"

bad_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)

        try:
            img = Image.open(file_path)
            img.verify()   # check if image is valid
        except Exception:
            bad_files.append(file_path)

print("Bad files found:", len(bad_files))

for file_path in bad_files:
    print("Removing:", file_path)
    os.remove(file_path)

print("All bad image files removed successfully!")

Bad files found: 1
Removing: /content/drive/MyDrive/AI-ML/FruitinAmazon/test/guarana/download (2).jpeg
All bad image files removed successfully!


In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    train,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    train,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 29 images belonging to 6 classes.


In [14]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [15]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.1528 - loss: 2.4689 - val_accuracy: 0.4444 - val_loss: 1.3955
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.3889 - loss: 1.5500 - val_accuracy: 0.5556 - val_loss: 1.0859
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6250 - loss: 0.9566 - val_accuracy: 0.8333 - val_loss: 0.7497
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8194 - loss: 0.6107 - val_accuracy: 0.8333 - val_loss: 0.5915
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step - accuracy: 0.8333 - loss: 0.4409 - val_accuracy: 0.8333 - val_loss: 0.4745
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8472 - loss: 0.4225 - val_accuracy: 0.9444 - val_loss: 0.3763
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.9167 - loss: 0.2511 - val_accuracy: 0.8889 - val_loss: 0.3996
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.9444 - loss: 0.1923 - val_accuracy: 0.8333 - val_loss: 0.4287
Epoch 9/10
3/3 

In [16]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9310 - loss: 0.2707
Test Accuracy: 0.931034505367279


In [17]:
predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
              precision    recall  f1-score   support

        acai       0.83      1.00      0.91         5
     cupuacu       1.00      1.00      1.00         5
    graviola       1.00      1.00      1.00         5
     guarana       1.00      1.00      1.00         4
     pupunha       0.83      1.00      0.91         5
      tucuma       1.00      0.60      0.75         5

    accuracy                           0.93        29
   macro avg       0.94      0.93      0.93        29
weighted avg       0.94      0.93      0.93        29



In [20]:
img_path = "/content/drive/MyDrive/AI-ML/FruitinAmazon/train/pupunha/images (1).jpeg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)
pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: pupunha


In [21]:
scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 4s/step - accuracy: 0.1667 - loss: 12.8530 - val_accuracy: 0.1667 - val_loss: 10.8936
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - accuracy: 0.2083 - loss: 6.5792 - val_accuracy: 0.3333 - val_loss: 3.8575
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.3333 - loss: 2.4081 - val_accuracy: 0.2778 - val_loss: 1.9395
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.5139 - loss: 1.4506 - val_accuracy: 0.3889 - val_loss: 1.4753
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.7222 - loss: 1.0567 - val_accuracy: 0.5556 - val_loss: 1.3467
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.8750 - loss: 0.6049 - val_accuracy: 0.2778 - val_loss: 2.0220
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.8750 - loss: 0.4017 - val_accuracy: 0.5556 - val_loss: 1.2247
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.9306 - loss: 0.2937 - val_accuracy: 0.3889 - val_loss: 1.3227
Epoch 9/1

In [22]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 885ms/step
              precision    recall  f1-score   support

        acai       0.62      1.00      0.77         5
     cupuacu       0.29      0.40      0.33         5
    graviola       0.57      0.80      0.67         5
     guarana       0.67      0.50      0.57         4
     pupunha       0.50      0.40      0.44         5
      tucuma       0.00      0.00      0.00         5

    accuracy                           0.52        29
   macro avg       0.44      0.52      0.46        29
weighted avg       0.43      0.52      0.46        29



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
img_path = "/content/drive/MyDrive/AI-ML/FruitinAmazon/train/pupunha/images (1).jpeg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step
Prediction: guarana
